# 08 Ask in plain words

*ASSIST*

**Question:** Can an AI drive CEPT without deciding the result?

- **You will see:** a saved agent session, one blocked command, and the checks that still decide.
- **Recorded, not live:** the cells replay a saved OpenCode session. No model is called and no key is needed.
- **Boundary:** the transcript is a narrated observation. Only solver artifacts and CEPT checks support engineering claims.

## 🧩 Inputs — the request

The request was one plain sentence. The saved session names the tool and model, and records that no credential value was kept.

In [1]:
import json

REPLAY = json.loads(r'''{
  "schema": "cept-public-agent-observe-replay-v1",
  "label": "SANITIZED_OBSERVE_REPLAY",
  "evidence_role": "agent interaction example only; not a solver receipt or product-acceptance result",
  "replayed_on": "2026-10-04, against the verb-only grammar",
  "replay_note": "Re-recorded after the verb-only migration. The previous version drove `cept study demo load-flow`, a route that no longer dispatches, so an agent copying it would now fail closed. Every command, exit status and digest below comes from this run, not from the earlier one.",
  "executor": {
    "tool": "OpenCode",
    "mode": "headless `opencode run --command cept` from a public-only tree",
    "model": "opencode/muse-spark-1.3-contributor-free",
    "variant": "xhigh",
    "credential_values_recorded": false
  },
  "request": "Run the bounded IEEE 13-node load-flow demonstration and explain only verified artifacts.",
  "observed_actions": [
    {
      "kind": "command",
      "command": "cept run public/examples/first_circuit_case.json",
      "exit_status": 2,
      "note": "CLI failed closed: the output directory is required, so nothing was written where the agent did not ask for it. The agent read the refusal and retried with an explicit output directory."
    },
    {
      "kind": "command",
      "command": "cept run public/examples/first_circuit_case.json --out runs/ieee13-demo --force",
      "exit_status": 0
    },
    {
      "kind": "command",
      "command": "cept verify runs/ieee13-demo",
      "exit_status": 0
    },
    {
      "kind": "read",
      "path": "runs/ieee13-demo"
    },
    {
      "kind": "read",
      "path": "runs/ieee13-demo/public-verification.json"
    },
    {
      "kind": "read",
      "path": "runs/ieee13-demo/validation_report.json"
    }
  ],
  "observed_verdict": {
    "study_type": "load_flow",
    "engine": "opendss",
    "passed": true,
    "status": "PASS",
    "claim": "WORKFLOW_VALIDATED",
    "case_fingerprint": "aaea341ddd26",
    "artifact_set_digest": "cept-artifacts-43fe433d80f945ee816d3dd8c426c907a8ab73dc2aba4911f12b705361ea43fc"
  },
  "sanitization": {
    "absolute_paths_replaced": true,
    "credentials_removed": true,
    "raw_model_events_not_published": true,
    "solver_values_recalculated": false
  },
  "claim_boundary": "The replay proves that this observed agent invoked the short `cept` commands and read their artifacts, including recovery from a fail-closed CLI error. Only the original solver artifacts support engineering claims. WORKFLOW_VALIDATED is not project or field validation."
}''')

print("Label     ", REPLAY["label"])
print("Request   ", REPLAY["request"])
print("Tool      ", REPLAY["executor"]["tool"])
print("Model     ", REPLAY["executor"]["model"])
print("Credential values recorded:", REPLAY["executor"]["credential_values_recorded"])


Label      SANITIZED_OBSERVE_REPLAY
Request    Run the bounded IEEE 13-node load-flow demonstration and explain only verified artifacts.
Tool       OpenCode
Model      opencode/muse-spark-1.3-contributor-free
Credential values recorded: False


## ▶ Run — what the agent did

Read the recorded actions. The first command was refused: the CLI failed closed because no output folder was given. The agent read the error and tried again.

In [2]:
#@title 2. What the agent did
for number, action in enumerate(REPLAY["observed_actions"], start=1):
    what = action.get("command") or action["path"]
    print(f"{number}. {action['kind']:<8} {what}")
    if action.get("exit_status") is not None:
        print(f"   exit status: {action['exit_status']}")
    if action.get("note"):
        print(f"   note: {action['note']}")

1. command  cept run public/examples/first_circuit_case.json
   exit status: 2
   note: CLI failed closed: the output directory is required, so nothing was written where the agent did not ask for it. The agent read the refusal and retried with an explicit output directory.
2. command  cept run public/examples/first_circuit_case.json --out runs/ieee13-demo --force
   exit status: 0
3. command  cept verify runs/ieee13-demo
   exit status: 0
4. read     runs/ieee13-demo
5. read     runs/ieee13-demo/public-verification.json
6. read     runs/ieee13-demo/validation_report.json


## ✓ Verify — what decides

The agent's words are not the evidence. The saved run is. The values below are copied from the saved session, which recorded them from CEPT's own verification files that the agent read.

In [3]:
#@title 3. What decides: the saved verdict
verdict = REPLAY["observed_verdict"]
for key in ("engine", "study_type", "status", "claim", "case_fingerprint", "artifact_set_digest"):
    print(f"{key:<20}{verdict[key]}")
print()
print("Claim boundary:")
print(REPLAY["claim_boundary"])
print()
print("Solver values recalculated for this page:", REPLAY["sanitization"]["solver_values_recalculated"])

engine              opendss
study_type          load_flow
status              PASS
claim               WORKFLOW_VALIDATED
case_fingerprint    aaea341ddd26
artifact_set_digest cept-artifacts-43fe433d80f945ee816d3dd8c426c907a8ab73dc2aba4911f12b705361ea43fc

Claim boundary:
The replay proves that this observed agent invoked the short `cept` commands and read their artifacts, including recovery from a fail-closed CLI error. Only the original solver artifacts support engineering claims. WORKFLOW_VALIDATED is not project or field validation.

Solver values recalculated for this page: False


## 🧾 Recipe — make it repeatable

A recipe is a fixed, versioned list of CEPT operations with declared inputs and outputs. It would give an agent, or you, the same path every time, and it decides nothing itself.

```yaml
# Trimmed sketch of the recipe format. Not runnable here.
schema: workflow-recipe-v1
id: study-evidence
inputs:
  - name: case
    type: file
stages:
  - id: check-case
    operation: check.case
    owner: python
    on_success:
      next: run-study
    on_failure:
      next: null          # a blocked Case stops the run
  - id: run-study
    operation: run.study
  - id: verify-run
    operation: verify.study
```

The recipe runtime does ship in the public wheel, so you can run one from here:
`cept run --recipe run-audit --input run_dir=<run-dir> --input case=<case.json> --input audit_root=<root> --out <out>`. Lesson 09 writes and runs one.

## ◇ Interpret — what this does and does not show

- **Supports:** an AI can issue the same short `cept` commands you would, and a refused command stays visible in the record.
- **Does not prove:** that the model reasoned correctly, that another model or run would behave the same, or that the results fit a real project.
- **Boundary:** `WORKFLOW_VALIDATED` is not project or field validation.

## 🧪 Optional — try it with your own model

Running an agent live needs a model provider you sign in to yourself. Never paste a key into a notebook cell, a prompt or a saved output. Follow the credential-safety notes in [public/agent/README.md](https://github.com/sarutesri/cept-studio-edu/blob/main/public/agent/README.md) and keep the key in your provider's own credential store.

Check the provider's data policy first. The model used in this recording is offered at a discount in exchange for permission to train on prompts and completions, according to OpenCode's published terms, and a live agent can send your workspace files to its provider.